# Grocery item GHG labeling — data preparation

This notebook builds the dataset behind the project: 75 branded grocery products pulled from [USDA FoodData Central](https://fdc.nal.usda.gov/), cleaned, and enriched with greenhouse-gas intensity estimates (kg CO₂e per kg) from Our World in Data, CarbonCloud, and the other sources cited at the bottom.

**Pipeline**

1. Search FoodData Central to hand-pick product `fdcId`s across high- and low-impact food groups
2. Bulk-fetch the selected products and trim to the useful columns
3. Export the cleaned table, enrich it with CO₂e intensities, and compute per-package emissions

> **Note:** cell outputs reflect the original February 2024 data pull. Re-running the API cells requires a free [FoodData Central API key](https://fdc.nal.usda.gov/api-key-signup.html). Copy `.env.example` to `.env` at the repo root and set `FDC_API_KEY=your-key`; the next cell loads it automatically. USDA records may have changed since the original pull.

In [ ]:
import os

import pandas as pd
import requests
from dotenv import load_dotenv

# Load secrets (FDC_API_KEY) from a local .env file at the repo root.
# Copy .env.example to .env and add your free key to re-run the API cells:
# https://fdc.nal.usda.gov/api-key-signup.html
load_dotenv()

In [ ]:
# Search FoodData Central to find fdcIds for products of interest.
# This example searches branded "Pita Chips"; the same search was repeated
# across food groups (meat, dairy, produce, snacks, ...) to hand-pick the
# items listed in the next cell.

API_KEY = os.environ["FDC_API_KEY"]

url = "https://api.nal.usda.gov/fdc/v1/foods/list"

params = {
    "api_key": API_KEY,
    "query": "Pita Chips",
    "dataType": "Branded",
    "pageSize": "10",
    "pageNumber": "1",
}

response = requests.get(url, params=params)
response.raise_for_status()

df_search = pd.DataFrame(response.json())
df_search

# Selected fdcIDs:

**High-impact CO2e**

* Protein -
    * Red Meat - 
Beef - 1597055; 
Impossible Beef - 2664238; 
Beyond Beef - 2663629; 
Lamb - 1568052; 
Pork Sausage - 2389332; 
Impossible Sausage - 2545686;
    * Poultry - 
Chicken - 2683232; 
Beyond Chicken - 1854765; 
Eggs - 2096834;
Just Egg - 2553551; 
    * Fish - 
Shrimp (Wild) - 2426953; 
Shrimp (Farmed) - 2313609; 
Salmon Filet (Wild) - 1578041; 
Tilapia Filet (Farmed) - 1457189; 
Tuna Canned  - 2052398; 
    * Other - 
Black Beans - 2425082; 
Peas - 1906028; 
Lentils - 2020249; 
Tofu - 1863123; 
Tempeh - 2596799


* Dairy -
    * Cheese - 
Cheddar - 2057196; 
American - 1592891; 
Parmesan - 2615037; 
PB-Cheddar - 2628069; 
PB-American - 2628070; 
PB-Parmesan - 2628072 
    * Butter - 
Regular - 2094280; 
Plant-based - 1493586 
    * Cream Cheese - 
Regular - 2094316; 
Plant-Based - 1989964 
    * Mayo - 
Regular - 1593206; 
Vegan - 2501701 
    * Milk - 
Whole - 2617014; 
Lowfat 1% - 2052228; 
Skim - 2402248; 
Oat - 2506626; 
Almond - 2309058; 
Soy - 2675467
    * Coffee Creamer - 
Regular - 1063145
Plant-based - 2102063
    * Ice Cream - 
Regular - 2121094
Plant-based - 2447124
    * Chocolate - 
Milk - 2032501
Dark - 2479821 
    * Frozen - 
Nuggets - 2548207
Vegan Nuggets - 2683081
Pizza - 2097032
Vegan Pizza - 2607541


* Caffeine -
Coffee - 2623658; 
Tea - 2032205; 

**Low-impact CO2e** 

* Produce - 
    * Veggies - 
Tomato - 2339607; 
Onion - 2438059; 
Lettuce - 2286527; 
Kale - 2391372
    * Fruit - 
Blueberries - 2395846; 
Orange - 1864680; 
Apple - 2117388; 
Banana - 2012128

* Carbs/Starch - 
Potatoes - 2251779; 
Rice - 2443255; 
Pasta - 1630363; 
Wheat Bread - 2054998; 
White Bread - 2054962; 
Couscous - 1882842

* Snacks - 
Tortilla chips - 1459670; 
Potato Chips - 2096564; 
Salsa - 1457440; 
Pita Chips - 2036533; 
Hummus - 2396908; 
Almonds - 2094362; 
Walnuts - 2009462; 
Cashews - 2384270; 
Peanuts - 1975864; 
Granola - 2131337; 
Granola Bar - 2657897; 
Dates - 2580894

In [ ]:
# All selected fdcIds in one list for the bulk request
fdc_ids = [
    "2580894", "2657897", "2396908", "2036533", "2096564", "2131337", "1975864", "2384270", "2009462", "2094362",
    "1457440", "1459670", "1882842", "2054962", "2054998", "1630363", "2443255", "2251779", "2286527", "2391372",
    "2012128", "2117388", "1864680", "2395846", "2438059", "2339607", "2607541", "2097032", "2683081", "2548207",
    "2479821", "2032501", "2447124", "2121094", "2102063", "1063145", "2675467", "2309058", "2506626", "2402248",
    "2052228", "2617014", "2501701", "1593206", "1989964", "2094316", "1493586", "2094280", "2628072", "2628070",
    "2628069", "2615037", "1592891", "2057196", "2032205", "2623658", "2553551", "2545686", "2664238", "1854765",
    "2663629", "2596799", "1863123", "2020249", "1906028", "2425082", "2052398", "1578041", "1457189", "2313609",
    "2426953", "2389332", "1568052", "2096834", "2683232", "1597055"
]

In [ ]:
# Bulk-fetch the full record for every selected product

url = "https://api.nal.usda.gov/fdc/v1/foods"

params = {
    "api_key": API_KEY,
    "fdcIds": ",".join(fdc_ids),
}

response = requests.get(url, params=params)
response.raise_for_status()

df_final = pd.DataFrame(response.json())
df_final.head()

In [ ]:
# Snapshot the raw pull
df_final.to_csv("../data/df_final.csv", index=False)

In [8]:
df_final.columns

Index(['discontinuedDate', 'foodComponents', 'foodAttributes', 'foodPortions',
       'fdcId', 'description', 'publicationDate', 'foodNutrients', 'dataType',
       'foodClass', 'shortDescription', 'modifiedDate', 'availableDate',
       'brandOwner', 'brandName', 'dataSource', 'brandedFoodCategory',
       'gtinUpc', 'householdServingFullText', 'ingredients', 'marketCountry',
       'servingSize', 'servingSizeUnit', 'packageWeight', 'foodUpdateLog',
       'labelNutrients', 'subbrandName', 'notaSignificantSourceOf',
       'gpcClassCode', 'preparationStateCode'],
      dtype='object')

In [9]:
# Remove unneeded columns 
df_final_clean = df_final.drop(['discontinuedDate', 'foodComponents', 'foodAttributes', 'foodPortions',
                                'foodUpdateLog','notaSignificantSourceOf', 'subbrandName','gpcClassCode', 
                                'preparationStateCode','gtinUpc','foodClass'],axis=1)

df_final_clean.head()

,fdcId,description,publicationDate,foodNutrients,dataType,shortDescription,modifiedDate,availableDate,brandOwner,brandName,dataSource,brandedFoodCategory,householdServingFullText,ingredients,marketCountry,servingSize,servingSizeUnit,packageWeight,labelNutrients
0,2580894,DATES,6/29/2023,"[{'type': 'FoodNutrient', 'nutrient': {'id': 1...",Branded,,4/27/2023,4/27/2023,"Dole Packaged Foods, LLC",DOLE,LI,Wholesome Snacks,,DATES.,United States,40.0,GRM,8 oz/227 g,"{'fat': {'value': 0.0}, 'saturatedFat': {'valu..."
1,2657897,"CHEWY CHOCOLATE GRANOLA, CHOCOLATE",10/26/2023,"[{'type': 'FoodNutrient', 'nutrient': {'id': 1...",Branded,,9/26/2023,9/26/2023,The Quaker Oats Company,QUAKER,LI,Cereal,3/4 cup (53g),"WHOLE GRAIN OATS, BROWN RICE CRISP (WHOLE GRAI...",United States,53.0,GRM,12.6 oz,"{'fat': {'value': 6.0}, 'saturatedFat': {'valu..."
2,2396908,HUMMUS,12/22/2022,"[{'type': 'FoodNutrient', 'nutrient': {'id': 1...",Branded,,4/8/2020,4/8/2020,"Wegmans Food Markets, Inc.",WEGMANS,LI,Dips & Salsa,1 CONTAINER,"CHICKPEAS, FILTERED WATER, TAHINI (SESAME SEED...",United States,57.0,g,24 oz/684 g,"{'fat': {'value': 8.0}, 'saturatedFat': {'valu..."
3,2036533,PITA CHIPS,10/28/2021,"[{'type': 'FoodNutrient', 'nutrient': {'id': 1...",Branded,NaN,6/25/2017,6/25/2017,The Kroger Co.,PRIVATE SELECTION,LI,Other Snacks,1 ONZ,"ENRICHED WHEAT FLOUR (WHEAT FLOUR, NIACIN, RED...",United States,28.0,g,NaN,"{'fat': {'value': 4.0}, 'saturatedFat': {'valu..."
4,2096564,POTATO CHIPS,10/28/2021,"[{'type': 'FoodNutrient', 'nutrient': {'id': 1...",Branded,NaN,3/12/2019,3/12/2019,Cape Cod Potato Chips Inc.,CAPE COD,LI,"Chips, Pretzels & Snacks",1 PACKAGE,"POTATOES, VEGETABLE OIL (CONTAINS ONE OR MORE ...",United States,NaN,g,2 oz/56.7 g,{}


In [ ]:
# Title-case the all-caps text fields for readability
text_cols = ['description', 'brandOwner', 'brandName', 'ingredients', 'servingSizeUnit', 'householdServingFullText']
df_final_clean[text_cols] = df_final_clean[text_cols].apply(lambda x: x.str.title())

df_final_clean.head()

In [ ]:
# Export the cleaned table — this is the file that gets hand-enriched
# with CO2e intensities (data/df_final_clean_enriched.csv)
df_final_clean.to_csv("../data/df_final_clean.csv", index=False)

In [ ]:
# Load the enriched dataset.
# Enrichment was curated by hand from the sources cited below and adds:
#   kgCO2e          - GHG intensity (kg CO2e per kg of food)
#   sourceCO2e      - citation for the chosen intensity estimate
#   packageGrams    - package weight parsed to grams
#   CO2eqPerPackage - kgCO2e x package weight in kg

df_enriched = pd.read_csv("../data/df_final_clean_enriched.csv")
df_enriched.head()

# From Tableau to the web

Version 1 of this project visualized these exports in Tableau — the storyboard PDF is preserved in `legacy/`, and the repo state at that point is tagged [`v1-tableau`](../../releases/tag/v1-tableau). Version 2 renders the same enriched dataset as an interactive site, built from `site/` in this repo.

# Citations:


**USDA API:** https://fdc.nal.usda.gov/api-guide.html


**Primary CO2e Data:**
https://ourworldindata.org/environmental-impacts-of-food#explore-data-on-the-environmental-impacts-of-food 
*Poore, J., & Nemecek, T. (2018). Reducing food’s environmental impacts through producers and consumers. Science. [original data].
* https://www.science.org/doi/10.1126/science.aaq0216#supplementary-materials
* https://ourworldindata.org/grapher/ghg-per-kg-poore
* https://ourworldindata.org/grapher/food-emissions-supply-chain
* https://ourworldindata.org/grapher/environmental-footprint-milks
* https://ourworldindata.org/grapher/ghg-emissions-seafood
* https://ourworldindata.org/food-choice-vs-eating-local


**Secondary CO2e Data:**
https://apps.carboncloud.com/climatehub/


**Other CO2e Data:**
* Alternative Beef: https://consumerecology.com/
* Chocolate: https://link.springer.com/article/10.1007/s11367-020-01817-6
* Nuts https://www.healabel.com/carbon-footprints-of-food-list/
* Just Egg https://www.ju.st/learn


**Climate Footprint Label Example - Oatly:**
 https://www.bloomberg.com/news/articles/2023-01-31/oatly-launches-climate-footprint-labels-in-the-us